# Module 05 — Tools

A model only predicts text — it can't truly do math, check the weather, or read your database. **Tools** fix that: you hand the model functions it's allowed to call.

The key idea (surprising at first): when the model "uses a tool," it doesn't run anything. It returns a **structured request** — "please call `multiply` with a=12, b=7." *You* run the function and give the answer back. That request → run → answer loop is what an agent automates later.

Run top to bottom.

## 0. Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


## 1. Define a tool

A tool is a normal Python function with the `@tool` decorator. Two things matter a lot:

- The **docstring** — the model reads it to decide *when* to use the tool. Write it like an instruction.
- The **type hints** — they become the tool's input schema (same idea as Module 04).

Here are two simple tools.

In [2]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Pretend this calls a real weather API. Hard-coded for the lesson.
    return f"It's 22°C and sunny in {city}."

print(multiply.name, "->", multiply.description)
print(get_weather.name, "->", get_weather.description)

multiply -> Multiply two numbers together.
get_weather -> Get the current weather for a given city.


## 2. Bind tools to the model

`model.bind_tools([...])` gives the model a *menu* of tools it may call. It doesn't force it — the model decides whether a question needs one.

Ask a plain question first: no tool needed, so you get a normal answer (empty `tool_calls`).

In [3]:
model_with_tools = model.bind_tools([multiply, get_weather])

plain = model_with_tools.invoke("Say hello in one word.")
print("Text:", plain.text)
print("Tool calls:", plain.tool_calls)   # empty -> the model just answered

Text: Hello.
Tool calls: []


## 3. The model requests a tool call

Now ask something that needs a tool. The model returns a message whose `.text` is usually empty, but whose **`.tool_calls`** lists what it wants run: the tool `name`, the `args`, and an `id`.

Read this carefully: nothing has been calculated yet. The model only *asked*.

In [4]:
response = model_with_tools.invoke("What is 12 multiplied by 7?")

print("Text:", repr(response.text))      # likely empty
print("Tool calls:", response.tool_calls)

Text: ''
Tool calls: [{'name': 'multiply', 'args': {'a': 12, 'b': 7}, 'id': 'c44bc659-f199-489c-ae18-e172baca5c14', 'type': 'tool_call'}]


## 4. Run the tool and return the result

Three steps to finish the job:
1. Keep the conversation in a `messages` list (the model's request must stay in history).
2. For each requested call, run the matching tool and wrap its result in a **`ToolMessage`** (tagged with the call's `id`).
3. Invoke the model **again** with the updated history — now it has the answer and writes a natural reply.

This is the full round trip.

In [5]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {"multiply": multiply, "get_weather": get_weather}

messages = [HumanMessage("What is 12 multiplied by 7?")]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)   # 1. keep the model's tool request in history

for call in ai_msg.tool_calls:                       # 2. run each requested tool
    chosen = tools_by_name[call["name"]]
    result = chosen.invoke(call["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    print(f"Ran {call['name']}{call['args']} -> {result}")

final = model_with_tools.invoke(messages)            # 3. ask again, now with the result
print("Final answer:", final.text)

Ran multiply{'b': 7, 'a': 12} -> 84
Final answer: 12 multiplied by 7 is 84.


## 5. The model picks the right tool

You gave it two tools. Ask a weather question and it chooses `get_weather` on its own — you didn't tell it which to use. With more tools, the model routes to whichever fits. (When tasks need *several* tools in sequence, you'd loop step 4 until there are no more tool calls — that loop is exactly what `create_agent` does in Module 11.)

In [6]:
messages = [HumanMessage("What's the weather in Tokyo?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for call in ai_msg.tool_calls:
    chosen = tools_by_name[call["name"]]
    result = chosen.invoke(call["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    print(f"Model chose: {call['name']}{call['args']}")

print("Final answer:", model_with_tools.invoke(messages).text)

Model chose: get_weather{'city': 'Tokyo'}
Final answer: It's 22°C and sunny in Tokyo.


## Recap

- A **tool** is a function + a description the model reads. Use `@tool`; the **docstring** and **type hints** define when it's used and what it accepts.
- **`bind_tools([...])`** offers the model a menu — it decides whether to call one.
- A tool request comes back in **`.tool_calls`** (`name`, `args`, `id`). The model does **not** execute it.
- You run the tool, wrap the output in a **`ToolMessage`** (matching `tool_call_id`), and **invoke again** for the final answer.
- The model **chooses** the right tool for the question. Looping this until done = an **agent**.

## 🛠️ Try it yourself

1. Add a third tool (e.g. `add(a, b)`), bind it, and ask a question that needs it.
2. Ask a compound question: "What is 3 times 4, and what's the weather in Paris?" Inspect `tool_calls` — how many requests come back?
3. Change a tool's docstring to be vague and see if the model still picks it correctly.
4. Print `ai_msg.tool_calls[0]` and identify the `name`, `args`, and `id` fields.
5. **Stretch:** write a `while` loop that keeps running tools and re-invoking until `tool_calls` is empty — you've just built a mini-agent.

When you're done, say **"next"** and we'll build **Module 06 — Chains (LCEL)**.